**imports:**

In [1]:
from __future__ import annotations
import argparse
from pathlib import Path
from typing import Iterable, Optional

import cv2
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

**constants:**

In [2]:
RANDOM_STATE = 42
IMG_SIZE = 32
CENTERED_DIGIT_SIZE = 24

**Histogram of Oriented Gradients**

For 32x32 images this creates 1764 useful shape/edge features per digit.

In [3]:
HOG = cv2.HOGDescriptor(_winSize = (32, 32), _blockSize = (8, 8), _blockStride = (4, 4), _cellSize = (4, 4), _nbins = 9, )

**Function definitions:**

In [11]:
def read_grayscale(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None: raise FileNotFoundError(f"Could not read image: {path}")
    return img

def maybe_invert(img: np.ndarray) -> np.ndarray:
    border = np.concatenate([img[0, :], img[-1, :], img[:, 0], img[:, -1]])
    if border.mean() > img.mean(): return 255 - img
    return img

def center_digit(img: np.ndarray) -> np.ndarray:
    img = maybe_invert(img)
    blurred = cv2.GaussianBlur(img, (3, 3), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    ys, xs = np.where(binary > 0)
    if len(xs) == 0 or len(ys) == 0: return cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation = cv2.INTER_AREA)
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    crop = img[y1:y2 + 1, x1:x2 + 1]
    h, w = crop.shape
    scale = CENTERED_DIGIT_SIZE/max(h, w)
    new_w = max(1, int(round(w*scale)))
    new_h = max(1, int(round(h*scale)))
    resized = cv2.resize(crop, (new_w, new_h), interpolation = cv2.INTER_AREA)
    canvas = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)
    x_offset = (IMG_SIZE - new_w) // 2
    y_offset = (IMG_SIZE - new_h) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    return canvas

def preprocess_simple(img: np.ndarray) -> np.ndarray:
    img = maybe_invert(img)
    return cv2.GaussianBlur(img, (3, 3), 0)

def extract_hog_features(img: np.ndarray, preprocess: str = "simple") -> np.ndarray:
    if preprocess == "center": img = center_digit(img)
    elif preprocess == "simple": img = preprocess_simple(img)
    else: raise ValueError("preprocess must be 'center' or 'simple'")
    features = HOG.compute(img).reshape(-1)
    return features.astype(np.float32)

def image_path_for_train(data_dir: Path, image_id: int, label: int) -> Path: return data_dir / "train" / str(label) / f"{image_id}.png"

def image_path_for_test(data_dir: Path, image_id: int) -> Path: return data_dir / "test" / f"{image_id}.png"

def build_train_features(data_dir: Path, train_df: pd.DataFrame, preprocess: str) -> tuple[np.ndarray, np.ndarray]:

    X = []
    y = []
    total = len(train_df)

    for i, row in enumerate(train_df.itertuples(index = False), start = 1):
        image_id = int(row.Id)
        label = int(row.Category)
        img = read_grayscale(image_path_for_train(data_dir, image_id, label))
        X.append(extract_hog_features(img, preprocess = preprocess))
        y.append(label)
        if i % 2000 == 0 or i == total: print(f"processed training images: {i}/{total}")

    return np.vstack(X), np.array(y, dtype = np.int64)

def build_test_features(data_dir: Path, test_df: pd.DataFrame, preprocess: str) -> np.ndarray:

    X = []
    total = len(test_df)

    for i, row in enumerate(test_df.itertuples(index = False), start = 1):
        image_id = int(row.Id)
        img = read_grayscale(image_path_for_test(data_dir, image_id))
        features = extract_hog_features(img, preprocess = preprocess)
        X.append(features)
        if i % 500 == 0 or i == total: print(f"processed test images: {i}/{total}")

    return np.array(X, dtype = np.float32)

def make_model(model_name: str):
    model_name = model_name.lower()
    if model_name == "sgd_svm": return Pipeline([('scaler', StandardScaler()), ('classifier', SGDClassifier(loss = "hinge", alpha = 0.0001, max_iter = 1000, tol = 1e-4, random_state = RANDOM_STATE, )), ])
    raise ValueError("Unknown model. Only sgd_svm available")

**output execution**

In [12]:
def main() -> None:

    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", type = Path, default = Path("."))
    parser.add_argument("--model", type = str, default = "sgd_svm", choices = ["sgd_svm"])
    parser.add_argument("--test-size", type = float, default = 0.2)
    parser.add_argument("--quick", type = int, default = None, help = "Use only N labelled images for a quick test run.")
    parser.add_argument("--preprocess", type = str, default = "center", choices = ["simple", "center"], help = "simple is fast; center crops/resizes the digit but is slower.")
    parser.add_argument("--output", type = Path, default = Path("submission.csv"))
    parser.add_argument("--save-model", type = Path, default = Path("hindi_digit_model.joblib"))
    args = parser.parse_args([])
    data_dir = args.data_dir
    train_csv = data_dir / "train.csv"
    test_csv = data_dir / "test.csv"
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
    print(f"Loaded train.csv: {train_df.shape[0]} rows")
    print(f"Loaded test.csv: {test_df.shape[0]} rows")

    if args.quick is not None and args.quick < len(train_df):
        train_df, _ = train_test_split(train_df, test_size = args.quick, stratify = train_df["Category"], random_state = RANDOM_STATE)
        train_df = train_df.reset_index(drop = True)
        print(f"Quick mode enabled: using {len(train_df)} labelled images")

    print("\nExtracting HOG features from training images...")
    X, y = build_train_features(data_dir, train_df, preprocess = args.preprocess)
    print(f"\nFeature matrix: {X.shape}")
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = args.test_size, stratify = y, random_state = RANDOM_STATE)
    model = make_model(args.model)
    print(f"\nTraining validation model: {args.model}")
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    print(f"\nTrain accuracy: {train_acc:.4f}")
    print(f"\nValidation accuracy: {val_acc:.4f}")
    print(f"\nDifference: {train_acc - val_acc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_val, val_pred))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_val, val_pred))
    print("\nRetraining final model on all labelled data...")
    final_model = make_model(args.model)
    final_model.fit(X, y)
    print("\nExtracting HOG features from test images...")
    X_test = build_test_features(data_dir, test_df, preprocess = args.preprocess)
    print("\nPredicting test labels...")
    test_pred = final_model.predict(X_test).astype(int)
    submission = pd.DataFrame({"Id": test_df["Id"].astype(int), "Category": test_pred})
    submission.to_csv(args.output, index = False)
    joblib.dump(final_model, args.save_model)
    print(f"\nSaved predictions to: {args.output}")
    print(f"\nSaved trained model to: {args.save_model}")
    print(submission.head(10))

if __name__ == "__main__": main()

Loaded train.csv: 17000 rows
Loaded test.csv: 3000 rows

Extracting HOG features from training images...
processed training images: 2000/17000
processed training images: 4000/17000
processed training images: 6000/17000
processed training images: 8000/17000
processed training images: 10000/17000
processed training images: 12000/17000
processed training images: 14000/17000
processed training images: 16000/17000
processed training images: 17000/17000

Feature matrix: (17000, 1764)

Training validation model: sgd_svm


C:\Users\user\Group10\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(



Train accuracy: 0.9969

Validation accuracy: 0.9871

Difference: 0.0099

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       340
           1       0.99      0.99      0.99       340
           2       0.98      0.98      0.98       340
           3       0.98      0.96      0.97       340
           4       0.99      1.00      0.99       340
           5       0.99      1.00      0.99       340
           6       0.97      0.98      0.97       340
           7       0.98      1.00      0.99       340
           8       1.00      0.99      1.00       340
           9       0.99      0.98      0.99       340

    accuracy                           0.99      3400
   macro avg       0.99      0.99      0.99      3400
weighted avg       0.99      0.99      0.99      3400


Confusion matrix:
[[338   0   0   0   0   0   0   2   0   0]
 [  0 336   0   0   0   0   4   0   0   0]
 [  0   0 334   6   0   0   0   0   0  

C:\Users\user\Group10\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(



Extracting HOG features from test images...
processed test images: 500/3000
processed test images: 1000/3000
processed test images: 1500/3000
processed test images: 2000/3000
processed test images: 2500/3000
processed test images: 3000/3000

Predicting test labels...

Saved predictions to: submission.csv

Saved trained model to: hindi_digit_model.joblib
      Id  Category
0  56604         6
1  29396         3
2  43803         6
3  12313         0
4  10341         8
5  40355         3
6  41117         4
7  71324         7
8   6854         1
9  12761         8
